In [1]:
# Load cluster
import pandas as pd
import numpy as np

BEST_CLUSTER = 0

train_df = pd.read_parquet( "../logs/kmean/train_clustered_gen1.parquet" )

test_df = pd.read_parquet( "../logs/kmean/test_clustered_gen1.parquet" )

print( "(Rows, Columns)" )
print( "Train:", train_df.shape )
print( "Test:", test_df.shape )

(Rows, Columns)
Train: (500486, 116)
Test: (150912, 116)


In [2]:
def cluster_composition( df ):
    cluster_summary = df.groupby( "cluster" ).agg(
        rows=( "ticker", "count" ),
        unique_tickers=( "ticker", "nunique" ),
        unique_sectors=( "sector", "nunique" )
    )

    year_counts = (
        df.assign(year=df["date"].dt.year)
          .groupby(["cluster", "year"])
          .size()
          .unstack( fill_value=0 )
    )

    return cluster_summary.join( year_counts )


print( "TRAIN" )
train_cluster_summary = cluster_composition( train_df )
display( train_cluster_summary.sort_values( "rows", ascending=False ) )

print( "\nTEST" )
test_cluster_summary = cluster_composition( test_df )
display( test_cluster_summary.sort_values( "rows", ascending=False ) )

TRAIN


,rows,unique_tickers,unique_sectors,2016,2017,2018,2019
cluster,,,,,,,
1,335161,2928,12,53050,109572,99330,73209
0,165325,2540,12,21695,39360,54105,50165



TEST


,rows,unique_tickers,unique_sectors,2020,2021
cluster,,,,,
1,94469,2818,12,39697,54772
0,56443,2239,12,37654,18789


In [3]:
# Look at cluster VS average overall
TARGET_COLS = [
    # Absolute returns
    "future_ret_1d",
    "future_ret_1w",
    "future_ret_1m",
    "future_ret_6m",
    "future_ret_1y",
    "future_ret_3y",
    "future_ret_5y",

    # Relative returns
    "future_excess_1d",
    "future_excess_1w",
    "future_excess_1m",
    "future_excess_6m",
    "future_excess_1y",
    "future_excess_3y",
    "future_excess_5y"
]

ID_COLS = ["ticker", "date", "sector"]
JUNK_EXCLUDE = {
    "price",
    "sector_size",
    "rows",
    "sector_size_market",
    "rows_market"
}

SEC_EXCLUDE = {
    "sector_is_trending",

    "sec_avg_ret_1w",
    "sec_avg_ret_1m",
    "sec_avg_ret_6m",
    "sec_avg_ret_1y",
    "sec_avg_ret_3y",
    "sec_avg_ret_5y",

    "sec_avg_ret_1w_market",
    "sec_avg_ret_1m_market",
    "sec_avg_ret_6m_market",
    "sec_avg_ret_1y_market",
    "sec_avg_ret_3y_market",
    "sec_avg_ret_5y_market",

    "sec_positive_1y_trend_pct",
    "sec_positive_5y_trend_pct",
    "sec_breadth_positive_1y",
    "sec_breadth_positive_5y",

    "sec_ret_1y_dispersion",
    "sec_ret_5y_dispersion",
    "sec_ret_1y_dispersion_market",
    "sec_ret_5y_dispersion_market",

    "quality_score",
    "sector_Unknown",
}

OTHER_EXCLUDE = {
    "pe"
}

numeric_cols = train_df.select_dtypes( include=[np.number] ).columns.tolist()

numeric_cols.remove( "cluster" )

feature_cols = [
    c for c in numeric_cols
    if c not in ( set( ID_COLS ) | set( TARGET_COLS ) | JUNK_EXCLUDE | SEC_EXCLUDE | {c for c in train_df.columns if c.startswith("future_")} | OTHER_EXCLUDE )
]

overall = train_df[feature_cols].mean()

cluster = train_df.groupby("cluster")[feature_cols].mean()

print( "Comparison of cluster average to overall average, per feature")

for c in cluster.index:
    print("\nCLUSTER", c)

    diff = (
        cluster.loc[c] - overall
    ).sort_values(ascending=False)

    print(diff.head(15))

Comparison of cluster average to overall average, per feature

CLUSTER 0
beta_1y                          0.146941
sector_Energy                    0.038516
sector_Healthcare                0.027317
sector_Basic Materials           0.018808
sector_Consumer Cyclical         0.017408
sector_Technology                0.012874
sector_Communication Services    0.011979
spy_ret_3y                       0.007225
sector_Consumer Defensive        0.002988
sector_Industrials               0.000496
sec_avg_5y_trend_market         -0.000003
sec_avg_5y_trend                -0.000053
sec_avg_1y_trend_market         -0.000068
sec_avg_ret_1d_market           -0.000124
sec_avg_1y_trend                -0.000132
dtype: float64

CLUSTER 1
ret_5y                 0.399661
excess_ret_5y          0.385584
risk_adjusted_1y       0.362308
risk_adjusted_5y       0.316657
log_market_cap         0.240727
ret_3y                 0.221216
ret_1y                 0.139226
excess_ret_1y          0.133392
5y_drawdown    

In [4]:
print( "Comparison of features, cluster to cluster" )

feature_compare = train_df.groupby("cluster")[
    [
        "ret_1y",
        "ret_5y",
        "excess_ret_1y",
        "excess_ret_5y",
        "monotonic_score",
        "risk_adjusted_1y",
        "5y_drawdown"
    ]
].mean()

feature_compare

Comparison of features, cluster to cluster


,ret_1y,ret_5y,excess_ret_1y,excess_ret_5y,monotonic_score,risk_adjusted_1y,5y_drawdown
cluster,,,,,,,
0,-0.151186,-0.034544,-0.267255,-0.854343,0.430731,-0.772989,-0.514777
1,0.270291,1.175345,0.136561,0.312932,0.713183,0.323821,-0.117852


In [5]:
# Compare stock performance
def cluster_performance(df):
    return (
        df.groupby("cluster")
          .agg(
              stocks=("ticker", "nunique"),
              rows=("ticker", "count"),

              median_1y=("future_ret_1y", "median"),
              median_3y=("future_ret_3y", "median"),
              median_5y=("future_ret_5y", "median"),

              median_excess_1y=("future_excess_1y", "median"),
              median_excess_3y=("future_excess_3y", "median"),
              median_excess_5y=("future_excess_5y", "median"),

              positive_5y=("future_ret_5y", lambda x: (x > 0).mean()),
              beats_spy_5y=("future_excess_5y", lambda x: (x > 0).mean())
          )
          .sort_values("median_excess_5y", ascending=False)
    )

print("TRAIN")
cluster_performance(train_df)



TRAIN


,stocks,rows,median_1y,median_3y,median_5y,median_excess_1y,median_excess_3y,median_excess_5y,positive_5y,beats_spy_5y
cluster,,,,,,,,,,
1,2928,335161,0.052464,0.188439,0.298351,-0.072358,-0.303206,-0.572941,0.730974,0.219375
0,2540,165325,-0.006222,0.129932,0.184962,-0.130852,-0.396952,-0.690481,0.596262,0.247682


In [6]:
print("TEST")
cluster_performance(test_df)

TEST


,stocks,rows,median_1y,median_3y,median_5y,median_excess_1y,median_excess_3y,median_excess_5y,positive_5y,beats_spy_5y
cluster,,,,,,,,,,
0,2239,56443,0.297448,0.247923,0.642960,0.039125,-0.104295,-0.359617,0.742838,0.373917
1,2818,94469,0.080474,0.056357,0.288331,-0.088941,-0.281804,-0.648915,0.683049,0.232267


In [7]:
# Top 50 stocks in best cluster
def best_50( df ):

    best = df[
        df["cluster"] == BEST_CLUSTER
    ]

    stocks = (
        best.groupby("ticker")
        .agg(
            observations=("ticker","count"),
            avg_future_5y=("future_ret_5y","mean"),
            median_future_5y=("future_ret_5y","median"),
            avg_return_1y=("future_ret_1y","mean"),
            avg_beta=("beta_1y","mean"),
            avg_market_cap=("log_market_cap","mean")
        )
        .sort_values(
            "median_future_5y",
            ascending=False
        )
    )

    return stocks.head(50)

best_50( train_df )

,observations,avg_future_5y,median_future_5y,avg_return_1y,avg_beta,avg_market_cap
ticker,,,,,,
ENPH,30,159.823097,159.352897,2.341390,2.863870,19.305254
GRVY,27,26.492218,34.873605,2.739361,0.644872,17.415885
GMWKF,25,27.256269,30.108692,3.042369,-0.271555,18.795250
CYRX,39,22.785599,27.195980,2.234686,1.652005,18.707485
CHRD,151,24.492626,23.548472,3.123527,4.067695,19.441080
ARWR,65,19.713151,18.693694,2.945199,3.016333,19.730129
OPRX,59,16.018727,16.092263,1.357531,0.422459,17.773533
CAMT,12,15.153389,15.186893,0.997490,1.299086,18.483290
ATLC,115,15.396563,14.660633,0.307534,0.286459,17.442992


In [8]:
best_50( test_df )

,observations,avg_future_5y,median_future_5y,avg_return_1y,avg_beta,avg_market_cap
ticker,,,,,,
CLS,43,36.721943,36.100372,0.292760,2.092602,20.607870
INOD,5,32.642156,33.242857,3.999420,0.270008,17.672562
ESEA,28,28.095871,27.793097,7.505833,0.992592,16.453539
MSTR,14,26.983148,24.061030,3.773354,1.043003,22.244467
STRL,6,23.644282,23.967354,1.128667,1.609173,19.584456
SMCI,1,21.386901,21.386901,0.596726,1.479714,21.129659
MOD,19,19.828779,19.615922,1.112459,1.822093,19.647190
BBW,33,19.585286,17.952930,4.311734,1.538552,17.400069
UAN,35,17.254756,17.847776,6.308177,2.223957,17.827069


In [9]:
# Cluster stability

cluster_frequency = (
    df.groupby("ticker")["cluster"]
    .agg(
        lambda x: x.value_counts().index[0]
    )
)

cluster_frequency.value_counts()

best_stocks = cluster_frequency[
    cluster_frequency == best_cluster
]

best_stocks.head()

NameError: name 'df' is not defined

In [ ]:
# Winning cluster VS market

comparison = pd.DataFrame({
    "winning_cluster": test_df[test_df.cluster==BEST_CLUSTER]["future_ret_5y"],
    "all_stocks": test_df["future_ret_5y"],
    "SPY": test_df[test_df.ticker == "SPY"]["future_ret_5y"]
})

print( "Comparison of returns" )
comparison.describe()

Comparison of returns


,winning_cluster,all_stocks,SPY
count,65598.000000,150912.000000,51.000000
mean,0.968448,0.797057,0.963987
std,2.135491,1.972052,0.116254
min,-0.999578,-0.999578,0.719934
25%,-0.003783,-0.084476,0.861662
50%,0.533836,0.401812,0.968480
75%,1.296885,1.108734,1.057251
max,53.853503,55.104512,1.177245
